[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/therestiscode/notebooks/blob/main/03-float32-ceiling/ceiling.ipynb)

# I summed twenty million ones. I got 16,777,216.

A table of ones, two columns, twenty million rows. Each column should add up to
20,000,000. In `float32`, NumPy says 16,777,216, and it says it without a
warning.

Run the cells in order. Nothing here is simulated: it is the same table and the
same line of code as the short. The arrays are about 160 MB each, which is fine
on a free Colab runtime.

## Set up, and make NumPy loud

In [1]:
import warnings
import numpy as np

# If anything below raised a warning or a floating-point error, it would stop here.
warnings.simplefilter("error")
np.seterr(all="raise")

N = 20_000_000
print("numpy", np.__version__)

numpy 2.5.4


Both switches are on so that "no warning" means something. Any warning becomes
an exception, and so does any floating-point error NumPy checks for.

## Twenty million ones, two columns

In [2]:
table = np.ones((N, 2), dtype=np.float32)

print("shape, dtype     ", table.shape, table.dtype)
print("table.sum(axis=0)", table.sum(axis=0).tolist())

shape, dtype      (20000000, 2) float32
table.sum(axis=0) [16777216.0, 16777216.0]


Ones add up to the number of rows, so both totals should be 20,000,000.
Both are 16,777,216, and the cell finished without an exception.

## Delete the other column, run the same line

In [3]:
alone = np.delete(table, 1, axis=1)

print("shape                 ", alone.shape)
print("alone.sum(axis=0)     ", alone.sum(axis=0).tolist())
print("first column unchanged", bool((alone[:, 0] == table[:, 0]).all()))
print("ones that vanished    ", int(alone.sum(axis=0)[0]) - int(table.sum(axis=0)[0]))

shape                  (20000000, 1)
alone.sum(axis=0)      [20000000.0]
first column unchanged True
ones that vanished     3222784


Same ones, same `sum(axis=0)`. The only change is that the column lost its
neighbour, and now the total is right. 3,222,784 ones went missing before.

What the neighbour holds makes no difference. It only has to be there:

In [4]:
zeros_beside = np.column_stack([np.ones(N, dtype=np.float32), np.zeros(N, dtype=np.float32)])
print("ones beside zeros", zeros_beside.sum(axis=0).tolist())
del zeros_beside

print("three columns    ", np.ones((N, 3), dtype=np.float32).sum(axis=0).tolist())

ones beside zeros [16777216.0, 0.0]
three columns     [16777216.0, 16777216.0, 16777216.0]


## The wall at 16,777,216

16,777,216 is 2²⁴. A `float32` keeps 24 bits of precision, so above that
number it can no longer store every whole number. The next one it can hold is
16,777,218.

In [5]:
wall = np.float32(16_777_216)
one = np.float32(1)

print("2**24           ", 2**24)
print("wall + 1        ", float(wall + one))
print("wall + 1 == wall", bool(wall + one == wall))
print("wall + 2        ", float(wall + np.float32(2)))
print("gap above wall  ", float(np.spacing(wall)))

2**24            16777216
wall + 1         16777216.0
wall + 1 == wall True
wall + 2         16777218.0
gap above wall   2.0


Adding one to the wall rounds straight back to the wall. That is the IEEE
rounding rule doing its job, which is why NumPy has nothing to warn about.

Past the wall there are no odd numbers at all. Ask for one and you get an even
neighbour:

In [6]:
for n in [16_777_217, 16_777_219, 20_000_000]:
    print(f"float32({n:,})".ljust(22), f"{float(np.float32(n)):,.0f}")

float32(16,777,217)    16,777,216
float32(16,777,219)    16,777,220
float32(20,000,000)    20,000,000


## Why two columns hit it and one column doesn't

The table is stored row by row (C order), so the two values in a row sit next
to each other in memory. To total each column, NumPy walks down the rows and
adds each value into a running total: one, then two, then three. That running
total reaches 16,777,216, and every one added after that rounds back.

`np.cumsum` builds exactly that running total, so you can watch it stall:

In [7]:
running = np.cumsum(np.ones(N, dtype=np.float32))

print("after 16,777,216 ones", f"{float(running[16_777_215]):,.0f}")
print("after 16,777,217 ones", f"{float(running[16_777_216]):,.0f}")
print("after 20,000,000 ones", f"{float(running[-1]):,.0f}")
del running

after 16,777,216 ones 16,777,216
after 16,777,217 ones 16,777,216
after 20,000,000 ones 16,777,216


A single column is one unbroken run of memory, and there NumPy sums
differently. It adds the numbers in small blocks and then adds the blocks'
totals together (pairwise summation), so it never adds a lone one to a total
that has already reached the wall.

The short says "in halves". That is the idea in one picture, not NumPy's exact
block size, and it is enough to see why it works:

In [8]:
half = np.ones(N // 2, dtype=np.float32).sum()

print("one half          ", f"{float(half):,.0f}")
print("half below wall   ", bool(half < wall))
print("half + half       ", f"{float(half + half):,.0f}")

one half           10,000,000
half below wall    True
half + half        20,000,000


NumPy's own documentation for `np.sum` says this, if you know where to look:

In [9]:
doc = " ".join(np.sum.__doc__.split())
start = doc.index("This improved precision is always provided")
end = doc.index("fast axis in memory.") + len("fast axis in memory.")
print(doc[start:end])

This improved precision is always provided when no ``axis`` is given. When ``axis`` is given, it will depend on which axis is summed. Technically, to provide the best speed possible, the improved precision is only used when the summation is along the fast axis in memory.


"The fast axis in memory" is the one whose values sit next to each other. In a
two-column C-order table that is the row, not the column you are totalling.

## The fix: do the total in float64

In [10]:
print('table.sum(axis=0, dtype="float64")', table.sum(axis=0, dtype="float64").tolist())
print("bytes as float32                  ", f"{table.nbytes:,}")
print("bytes as float64                  ", f"{table.nbytes * 2:,}")

table.sum(axis=0, dtype="float64") [20000000.0, 20000000.0]
bytes as float32                   160,000,000
bytes as float64                   320,000,000


The data stays `float32`, so you keep the memory you chose it for. Only the
running total is kept in `float64`, which has room for whole numbers up to
2⁵³.

Keep your data in float32 if you want, but do the total in float64.

## "Who has twenty million ones?"

Nobody. The ones only make the arithmetic easy to check. Here are twenty million
prices between 1.00 and 100.00, with a quantity column beside them, which is an
ordinary sales table:

In [11]:
del table, alone
rng = np.random.default_rng(13)
price = np.round(rng.uniform(1, 100, N), 2).astype(np.float32)
qty = rng.integers(1, 5, N).astype(np.float32)
sales = np.column_stack([price, qty])

true = float(price.sum(dtype=np.float64))
in_table = float(sales.sum(axis=0)[0])
fixed = float(sales.sum(axis=0, dtype="float64")[0])

print(f"true total            {true:>18,.2f}")
print(f"sum(axis=0)           {in_table:>18,.2f}   off by {in_table - true:+,.2f}")
print(f"off by, percent       {100 * (in_table - true) / true:>+17.3f}%")
print(f'with dtype="float64"  {fixed:>18,.2f}   off by {fixed - true:+,.2f}')
del sales

true total              1,010,107,464.38
sum(axis=0)               972,393,024.00   off by -37,714,440.38
off by, percent                  -3.734%
with dtype="float64"    1,010,107,464.38   off by +0.00


Prices are not ones, so the total doesn't freeze, but it does the same thing in
a smaller way: once the running total is large, each new price is rounded to
the coarse steps `float32` has up there. Thirty-seven million short, and still
no warning.

## "It never gets there" is shorthand

On one column the total does reach 16,777,216 and pass it, at the very last
step, when the two big pieces are added together. It lands on 20,000,000
exactly only because 20,000,000 is even. Add one more row:

In [12]:
print("20,000,000 ones, one column", f"{float(np.ones(N, dtype=np.float32).sum()):,.0f}")
print("20,000,001 ones, one column", f"{float(np.ones(N + 1, dtype=np.float32).sum()):,.0f}")
print('20,000,001, dtype="float64"', f"{float(np.ones(N + 1, dtype=np.float32).sum(dtype='float64')):,.0f}")

20,000,000 ones, one column 20,000,000
20,000,001 ones, one column 20,000,000
20,000,001, dtype="float64" 20,000,001


So a single column is not safe either, just much less wrong. `dtype="float64"`
fixes both.

## Where the same call is fine

The trouble is the pairing of `float32`, a total along an axis, and that axis
not being the fast one in memory. Change any of the three and it goes away:

In [13]:
table = np.ones((N, 2), dtype=np.float32)

print("table.sum(), no axis        ", f"{float(table.sum()):,.0f}")
print("table[:, 0].sum()           ", f"{float(table[:, 0].sum()):,.0f}")
print('order="F", sum(axis=0)      ', np.ones((N, 2), dtype=np.float32, order="F").sum(axis=0).tolist())
print("np.ones(3).dtype, by default", np.ones(3).dtype)

table.sum(), no axis         40,000,000
table[:, 0].sum()            20,000,000
order="F", sum(axis=0)       [20000000.0, 20000000.0]
np.ones(3).dtype, by default float64


NumPy's default is `float64`, so you only meet this once something chose
`float32` for you: a file format, a GPU library, or a "save memory" line.

`float64` has the same wall, much further out:

In [14]:
print("float64 wall, 2**53", f"{2**53:,}")
print("wall + 1 == wall   ", bool(np.float64(2**53) + 1.0 == np.float64(2**53)))

float64 wall, 2**53 9,007,199,254,740,992
wall + 1 == wall    True


---

[@therestiscode](https://youtube.com/@therestiscode)